# 03 — SVM Modeling
### Online Shoppers Purchasing Intention — Support Vector Machine

This notebook covers the SVM portion of the classification comparison:
data loading, preprocessing, hyperparameter tuning, evaluation, and
interpretation. It mirrors the structure of `03_KNN_Modeling.ipynb` and
`03_XGBoost_Modelling.ipynb` so results can be compared consistently
across all three models on the **Model Comparison** page of the app.


In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import matplotlib.pyplot as plt

from src.data_preprocessing import preprocess_data
from src.utils import evaluate_model, print_metrics
from models.svm_model import (
    train_svm,
    get_grid_search_results,
    cross_validate_svm,
    get_svm_feature_importance,
    generate_svm_report,
    plot_svm_learning_curve,
)

%matplotlib inline
pd.set_option("display.max_columns", None)


## 1. Load & Split Data

Raw (untransformed) train/test DataFrames are used — scaling and
one-hot encoding happen *inside* the pipeline so SMOTE and
`GridSearchCV` never see leaked information from the test fold.


In [ ]:
DATA_PATH = project_root / "data" / "raw" / "online_shoppers_intention.csv"

X_train, X_test, y_train, y_test, _ = preprocess_data(
    filepath=str(DATA_PATH),
    outlier_method="none",
    test_size=0.2,
    transform=False,
    scale_numerical=True,  # SVM is distance-based -> features must be scaled
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train class balance:\n{y_train.value_counts(normalize=True)}")


## 2. Hyperparameter Tuning

We tune `C`, `kernel`, `gamma`, and `class_weight` via `GridSearchCV`
using **F1-score** (not accuracy) because the target is imbalanced
(~85% No-Purchase / 15% Purchase). SMOTE runs inside the pipeline so
oversampling only ever touches the training fold in each CV split.


In [ ]:
model, search_obj = train_svm(
    X_train,
    y_train,
    use_smote=True,
    cv=5,
    scoring="f1",
    output_path="../saved_models/svm_model.pkl",
    verbose=1,
)


In [ ]:
tuning_results = get_grid_search_results(search_obj)
tuning_results.head(10)


## 3. Held-out Test Set Evaluation

Standard point-estimate metrics on the 20% held-out test split.


In [ ]:
metrics = evaluate_model(model, X_test, y_test)
print_metrics("SVM Classifier", metrics)


## 4. K-Fold Cross-Validation of the Final Model

A single train/test split can be noisy on a dataset this size. Re-running
stratified 5-fold CV on the *final* tuned pipeline gives a mean +/- std
for each metric — more defensible for the Results section than a lone
point estimate.


In [ ]:
cv_summary = cross_validate_svm(model, X_train, y_train, cv=5)
cv_summary[["Mean", "Std"]]


## 5. Diagnostic & Interpretation Plots

Generates, in one call:
1. Confusion matrix
2. ROC curve
3. Precision-Recall curve (more informative than ROC given class imbalance)
4. C / gamma hyperparameter heatmap (rbf-kernel runs)
5. Feature importance (coefficients for a linear kernel, permutation
   importance otherwise)

All figures are saved to `report_assets/plots/` for direct use in the
Word/PDF report.


In [ ]:
report_metrics = generate_svm_report(
    model,
    X_test,
    y_test,
    search_obj=search_obj,
    X_importance=X_train,
    y_importance=y_train,
    save_dir="../report_assets/plots",
    show=True,
)


## 6. Learning Curve

Checks whether the model would benefit from more training data, or
whether it's already over/underfitting at the current dataset size.


In [ ]:
plot_svm_learning_curve(
    model, X_train, y_train, cv=5, scoring="f1",
    save_dir="../report_assets/plots", show=True,
)


## 7. Summary for the Report

Fill in the actual numbers once this notebook has been run against the
real dataset (this notebook ships without pre-run outputs since the
real `online_shoppers_intention.csv` wasn't available in this
environment — see the README note below).

| Metric | Value |
|---|---|
| Best hyperparameters | `search_obj.best_params_` |
| Test Accuracy | `metrics['Accuracy']` |
| Test Precision | `metrics['Precision']` |
| Test Recall | `metrics['Recall']` |
| Test F1 | `metrics['F1']` |
| Test AUC | `metrics['AUC']` |
| 5-fold CV F1 (mean +/- std) | `cv_summary.loc['F1', 'Mean']` +/- `cv_summary.loc['F1', 'Std']` |

**Discussion prompts to expand on in the report:**
- How does SMOTE affect precision vs. recall for the minority (Purchase) class?
- Which features dominate the SVM decision boundary, and does that match
  intuition (e.g. `PageValues` is typically the strongest predictor)?
- Does the rbf kernel meaningfully outperform linear here, or does the
  added complexity not pay off relative to KNN/XGBoost?
